# Rebuild the public USGS datasets

Re-creates `public.usgs_gages` and `public.usgs_streamflow` as the v1 USGS ingest needs them
(teehr-cloud-core#271), instead of adding columns to the copied ones:

| v1 dataset (`public`) | what |
| --- | --- |
| `usgs_gages` (layer) | every legacy USGS location, with the legacy location attributes as typed columns |
| `usgs_streamflow` (observed) | `approval_status` and `qualifier` columns; filled by `v1-backfill-usgs-streamflow` |
| `<CROSSWALK>` (crosswalk) | the legacy crosswalk from the gages to `<FORECAST>` |
| `usgs_streamflow_{percentiles,climatology,naive_scale}` | re-registered; rebuilt by the `v1-build-*` flows |

**Run it as a platform admin**, on KinD first, then use1 with the v1 schedules paused (deleting
empties every dataset above until the backfill and builds finish). Works on KinD (its legacy
warehouse) and use1.

**Re-runnable:** registering skips datasets that exist; the layer and crosswalk are replaced
whole. The deletes run only with `DELETE = True`.

Steps: settings, attribute types (writes nothing), delete, register and load, then the flows.

In [ ]:
import json
import os

import requests
from pyspark.sql import functions as F
from teehr.evaluation.spark_session_utils import create_spark_session

## 1. Settings

In [ ]:
# use1: the copied nwm30 configuration. KinD: its seeded nwm31 forecast.
CROSSWALK, FORECAST, PREFIX = "usgs_nwm30", "nwm30_medium_range", "nwm30"
# CROSSWALK, FORECAST, PREFIX = "usgs_nwm31", "nwm31_short_range", "nwm31"

GAGES, OBSERVED = "usgs_gages", "usgs_streamflow"
DERIVED = ("percentiles", "climatology", "naive_scale")
# Dependents first: the control plane won't delete a dataset others refer to.
OLD = [*(f"{OBSERVED}_{k}" for k in DERIVED), CROSSWALK, OBSERVED, "usgs_gauges", GAGES]
DELETE = False  # True to delete OLD (and their data)

# The control plane API, from inside the cluster. From outside: https://control-plane.teehr.rtiamanzi.org
# (use1), https://control-plane.teehr.local.app.garden (KinD).
CONTROL_PLANE = os.environ.get(
    "CONTROL_PLANE_URL", "http://teehr-control-plane.teehr-hub.svc.cluster.local:8000"
)
VERIFY_TLS = "local.app.garden" not in CONTROL_PLANE

LEGACY, PUBLIC = "iceberg.teehr", "iceberg.public"
SOURCE = "Legacy TEEHR warehouse (locations, location attributes, crosswalk); USGS Water Data"

In [ ]:
spark = create_spark_session()
spark.conf.set("spark.sql.iceberg.check-nullability", "false")
spark.conf.set("spark.sql.session.timeZone", "UTC")


def access_token():
    """Your access token, refreshed when it's about to expire (JupyterHub sets it)."""
    try:
        from teehr.utils.keycloak_auth import refresh_user_token_from_env
    except ImportError:
        return os.environ["POLARIS_USER_TOKEN"]
    access, _refresh = refresh_user_token_from_env()
    return access


api = requests.Session()
api.verify = VERIFY_TLS


def call(method, path, **kwargs):
    resp = api.request(method, f"{CONTROL_PLANE}/v1{path}",
                       headers={"Authorization": f"Bearer {access_token()}"}, **kwargs)
    if resp.status_code >= 400 and resp.status_code != 404:
        raise RuntimeError(f"{method} {path}: HTTP {resp.status_code}: {resp.text[:1000]}")
    return resp


def exists(name):
    return call("GET", f"/projects/public/datasets/{name}").status_code == 200


def register(body):
    if exists(body["name"]):
        print(f"public.{body['name']}: exists")
        return
    call("POST", "/projects/public/datasets", json=body)
    print(f"public.{body['name']}: registered")


me = call("GET", "/projects/public").json()
print(f"public: {me['status']}; your role there: {me.get('my_role')}")

## 2. Attribute types (writes nothing)

Legacy keeps every attribute as text. A column is `boolean` when every value is `True`/`False`,
`float64` when every value is a number without a leading zero (so codes like HUCs, `02...`, stay
text), else `string`. Names are lower-cased (Trino folds identifiers). Override in `TYPES`.

In [ ]:
# A type here wins: codes that look like numbers stay text; whole numbers as integers.
TYPES = {"epa_ecoregion_l1": "string", "epa_ecoregion_l2": "string",
         "first_year": "int32", "last_year": "int32", "num_timeseries": "int32",
         "stream_order": "int32"}

kinds = spark.sql(f'''
    SELECT attribute_name AS name,
           count(*) AS n,
           sum(CASE WHEN value IN ('True', 'False') THEN 1 ELSE 0 END) AS booleans,
           sum(CASE WHEN value RLIKE '^-?(0|[1-9][0-9]*)?([.][0-9]+)?([eE][-+]?[0-9]+)?$'
                         AND value RLIKE '[0-9]' THEN 1 ELSE 0 END) AS numbers
    FROM {LEGACY}.location_attributes
    WHERE location_id LIKE 'usgs-%' AND value IS NOT NULL
    GROUP BY 1 ORDER BY 1
''').toPandas()
kinds["type"] = [
    TYPES.get(r.name.lower())
    or ("boolean" if r.booleans == r.n else "float64" if r.numbers == r.n else "string")
    for r in kinds.itertuples()
]
kinds["column"] = kinds["name"].str.lower()
assert kinds["column"].is_unique, "attribute names collide once lower-cased"
attributes = [{"name": r.column, "type": r.type} for r in kinds.itertuples()]


def provenance(props):
    """Legacy properties as text: source; method; when retrieved or calculated."""
    if not props:
        return ""
    when = [f"{k.split('_')[1]} {v}" for k, v in props.items() if k.startswith("date_")]
    parts = [props.get("source"), props.get("method"), *when]
    return "; ".join(p for p in parts if p)


# Each attribute's description, then where its values came from (the legacy properties,
# the same for nearly all of an attribute's values: the most common one).
descriptions = dict(spark.sql(f"SELECT name, description FROM {LEGACY}.attributes").toPandas().values)
origins = spark.sql(f'''
    SELECT attribute_name AS name, to_json(properties) AS properties, count(*) AS n
    FROM {LEGACY}.location_attributes
    WHERE location_id LIKE 'usgs-%' AND cardinality(properties) > 0
    GROUP BY 1, 2
''').toPandas().sort_values("n").groupby("name").last()["properties"]  # a map can't be grouped
for a, name in zip(attributes, kinds["name"]):
    text = descriptions.get(name) or ""
    if "default description" in text:  # legacy's placeholder
        text = ""
    origin = provenance(json.loads(origins[name])) if name in origins.index else ""
    doc = f"{text} (source: {origin})" if text and origin else text or (f"Source: {origin}" if origin else "")
    if doc:
        a["doc"] = doc[:1000]
kinds["doc"] = [a.get("doc", "") for a in attributes]
kinds[["name", "column", "type", "n", "booleans", "numbers", "doc"]]

In [ ]:
gages = spark.sql(f'''
    SELECT count(*) AS locations,
           sum(CASE WHEN a.active AND a.iv THEN 1 ELSE 0 END) AS active_with_inst_discharge
    FROM {LEGACY}.locations l
    LEFT JOIN (
        SELECT location_id,
               max(CASE WHEN attribute_name = 'is_active' THEN value END) = 'True' AS active,
               max(CASE WHEN attribute_name = 'has_inst_discharge' THEN value END) = 'True' AS iv
        FROM {LEGACY}.location_attributes WHERE location_id LIKE 'usgs-%' GROUP BY 1
    ) a ON a.location_id = l.id
    WHERE l.id LIKE 'usgs-%'
''')
gages.show()

## 3. Delete the old datasets (`DELETE = True`)

In [ ]:
def refers_to_observed(dataset):
    """A crosswalk or derived dataset of OBSERVED (whatever it's called)."""
    refs = [*(dataset.get("crosswalk") or {}).get("left", "").split(","),
            (dataset.get("crosswalk") or {}).get("right", ""),
            (dataset.get("derived") or {}).get("observed", "")]
    return any(r.split(".")[-1] == OBSERVED for r in refs if r)


listed = call("GET", "/projects/public/datasets").json()
dependents = [d["name"] for d in listed if refers_to_observed(d) and d["name"] not in OLD]
for name in [*dependents, *OLD]:
    if not exists(name):
        print(f"public.{name}: not there")
    elif DELETE:
        call("DELETE", f"/projects/public/datasets/{name}")
        print(f"public.{name}: deleted")
    else:
        print(f"public.{name}: would delete (DELETE = False)")

## 4. Register and load

In [ ]:
located = spark.sql(f'''
    SELECT to_json(properties) AS properties, count(*) AS n FROM {LEGACY}.locations
    WHERE id LIKE 'usgs-%' AND cardinality(properties) > 0 GROUP BY 1 ORDER BY n DESC LIMIT 1
''').collect()
LAYER_SOURCE = (f"{SOURCE}. Locations: {provenance(json.loads(located[0].properties))}"
                if located else SOURCE)
register({"type": "layer", "name": GAGES, "source": LAYER_SOURCE,
          "description": "USGS stream gages", "attributes": attributes})

casts = []
for a, name in zip(attributes, kinds["name"]):
    v = f"max(CASE WHEN attribute_name = '{name}' THEN value END)"
    if a["type"] == "boolean":
        casts.append(f"{v} = 'True' AS `{a['name']}`")
    elif a["type"] == "string":
        casts.append(f"{v} AS `{a['name']}`")
    else:
        sql = {"float64": "DOUBLE", "int32": "INT", "int64": "BIGINT"}[a["type"]]
        casts.append(f"CAST({v} AS {sql}) AS `{a['name']}`")
layer = spark.sql(f'''
    WITH attrs AS (
        SELECT location_id, {", ".join(casts)}
        FROM {LEGACY}.location_attributes WHERE location_id LIKE 'usgs-%' GROUP BY 1
    )
    SELECT l.id AS location_key, l.name, l.geometry,
           ST_XMin(g) AS bbox_xmin, ST_YMin(g) AS bbox_ymin,
           ST_XMax(g) AS bbox_xmax, ST_YMax(g) AS bbox_ymax,
           {", ".join(f"a.`{a['name']}`" for a in attributes)}
    FROM (SELECT *, ST_GeomFromWKB(geometry) AS g FROM {LEGACY}.locations
          WHERE id LIKE 'usgs-%') l
    LEFT JOIN attrs a ON a.location_id = l.id
''')
layer.writeTo(f"{PUBLIC}.{GAGES}").overwrite(F.lit(True))
print(f"{GAGES}: {spark.table(f'{PUBLIC}.{GAGES}').count():,} gages")

In [ ]:
register({"type": "timeseries", "name": OBSERVED, "role": "observed",
          "variable": "streamflow", "statistic": "inst",
          "optional_columns": ["approval_status", "qualifier"], "location_layer": GAGES,
          "source": "USGS Water Data (api.waterdata.usgs.gov): instantaneous discharge, "
                    "00060, hourly",
          "description": "USGS instantaneous streamflow"})

In [ ]:
if exists(FORECAST):
    register({"type": "crosswalk", "name": CROSSWALK, "left": OBSERVED, "right": FORECAST,
              "source": SOURCE, "description": f"USGS gages to their {PREFIX} reaches"})
    links = spark.sql(f'''
        SELECT DISTINCT primary_location_id AS left_key, secondary_location_id AS right_key
        FROM {LEGACY}.location_crosswalks
        WHERE primary_location_id LIKE 'usgs-%' AND secondary_location_id LIKE '{PREFIX}-%'
    ''')
    links.writeTo(f"{PUBLIC}.{CROSSWALK}").overwrite(F.lit(True))
    print(f"{CROSSWALK}: {spark.table(f'{PUBLIC}.{CROSSWALK}').count():,} links")
else:
    print(f"public.{FORECAST} isn't there: no crosswalk")

for kind in DERIVED:
    register({"type": kind, "name": f"{OBSERVED}_{kind}", "observed": OBSERVED,
              "description": f"{kind.replace('_', ' ')} of {OBSERVED}"})

## 5. Fill them (Prefect)

1. **`v1-backfill-usgs-streamflow`**, Custom run with `where`: `has_inst_discharge == True`
   (every gage with instantaneous discharge, active or not: their whole record from the USGS
   bulk files, then the API from where the files end).
2. **`v1-build-usgs-streamflow-percentiles`**, `-climatology`, `-naive-scale`.
3. The daily **`v1-ingest-usgs-streamflow`** with `where`:
   `is_active == True and has_inst_discharge == True` (the legacy ingest's gages); then turn on
   its schedule.

Then check below.

In [ ]:
spark.sql(f'''
    SELECT count(DISTINCT series_key) AS series, count(*) AS rows,
           min(value_time) AS first, max(value_time) AS last,
           count_if(approval_status = 'Provisional') AS provisional
    FROM {PUBLIC}.{OBSERVED}
''').show(truncate=False)